# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shaikh-Ashmaan/MachineLearning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Chosen Lane:** Lane 2 — Refresh / Content Opportunity Scoring.

**ML Task Type:** **Binary classification paired with priority ranking (scoring)**.
* First, a supervised classifier estimates the probability $P(\text{decline} \mid X)$ that an eligible content item is experiencing or entering traffic decline based on observable search, engagement, and freshness signals.
* Second, those predicted probabilities are sorted into a **ranked review queue** (Top-$K$ prioritization) so editors review the highest-risk, highest-opportunity pages first rather than acting on a raw yes/no cutoff.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

* **Starter Dataset Proxy Target:** `is_declining_label = (trend_direction == "down")`, evaluated on mature visible content (`impressions_90d > 0` and `content_age_days >= 90`). Because `trend_direction` is computed from the current 90-day window, it acts as a **current-window proxy** rather than a true future outcome.
* **Warehouse Capstone Target (Leakage-Safe):** When scaling to the full `FlyRank/internship-warehouse` daily panel, the target will be a **future-window binary outcome**: whether a page experiences a sustained drop (e.g., $>20\%$ decline in rolling 30-day impressions or clicks) in the **next 30 days** given observable features from the **prior 90 days** (`prior 90d -> next 30d`), after filtering out low-volume noise and site-wide consolidation.
* **Action Supported:** Generating a Top-50 weekly editorial refresh queue with reason codes (e.g., stale high-visibility page, decaying page-one asset, low-CTR opportunity) to guide content updates, expansions, or consolidation reviews.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

* **Primary Metric — Precision@50 (and Precision@20):** Editorial teams have fixed weekly capacity (e.g., reviewing 20 to 50 pages per sprint). **Precision@50** directly measures operational value: *of the top 50 pages the model recommends reviewing first, what fraction are truly declining?*
* **Secondary Metrics — Average Precision (PR-AUC) & ROC AUC:** Because positive decline cases are a minority class and ranking order matters across the whole queue, **Average Precision** evaluates ranking quality across all thresholds without relying on a single probability cutoff.
* **Why Raw Accuracy Is Wrong Here:** If only ~25–30% of mature pages are declining, a trivial rule that predicts "not declining" for every page achieves ~70–75% accuracy while catching zero refresh candidates.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import os, subprocess, pandas as pd

# Ensure the starter repo/CSV is available in Colab
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Filter to Lane 2's decision scope: mature pages with observable search visibility
lane_df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset=["content_id"]).copy()

# Sketch the binary target column
lane_df["target_is_declining"] = (lane_df["trend_direction"] == "down").astype(int)

print(f"Unit of analysis: 1 row = 1 unique content item (content_id)")
print(f"Lane dataframe shape: {lane_df.shape[0]:,} rows x {lane_df.shape[1]} columns")
print(f"Target positive rate (target_is_declining == 1): {lane_df['target_is_declining'].mean():.1%} ({lane_df['target_is_declining'].sum():,} pages)\n")

# Display sample rows with key observable features and the sketched target
cols_to_show = [
    "content_id", "client_id", "impressions_90d", "clicks_90d",
    "avg_position", "ctr", "content_age_days", "days_since_last_update",
    "trend_direction", "target_is_declining"
]
lane_df[cols_to_show].head(5)

Unit of analysis: 1 row = 1 unique content item (content_id)
Lane dataframe shape: 30,000 rows x 45 columns
Target positive rate (target_is_declining == 1): 54.2% (16,262 pages)



,content_id,client_id,impressions_90d,clicks_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction,target_is_declining
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,0.76,187,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,0.05,445,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,141,20,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,0.49,463,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,0.13,263,14,down,1


## 5. Why ML beats a fixed rule here

* **Multi-Signal Trade-Offs:** A hand-written rule relies on rigid single-metric thresholds (e.g., `days_since_last_update >= 180` AND `impressions_90d >= 500`), treating every stale page identically regardless of its position tier, click-through rate, engagement rate, or intent type.
* **Non-Linear Interactions:** Search decay rarely stems from one variable alone—for instance, a 400-day-old evergreen guide holding position 2 with strong CTR behaves very differently from a 200-day-old page slipping from position 4 to position 9 with falling engagement. Tree-based models (Decision Trees, Random Forests) learn these conditional interactions directly from data.
* **Empirical Evidence from Starter Pipeline:** On client-holdout validation, the hand-written baseline achieved `0.240` Precision@50 (~12/50 correct), whereas a Random Forest classifier achieved `0.740` Precision@50 (~37/50 correct)—more than tripling the number of valid refresh candidates surfaced per 50 reviews.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.